# Image Denoising Using Convolutional Autoencoders

**Mini-Project — Neural Networks and Deep Learning**

This notebook trains a convolutional autoencoder to remove artificial Gaussian noise from MNIST digit images, evaluates it with MSE / PSNR / SSIM, and visualizes original vs. noisy vs. denoised images.

This notebook is **self-contained and runs start to finish in Google Colab** — it does not depend on the `src/` folder of the project (it redefines the same model/data/metrics logic inline), so you can open it fresh in Colab, run every cell in order, and get a trained model plus all evaluation outputs.

**Runtime:** CPU works fine for this small model/dataset (a few minutes total). For a small speedup: `Runtime > Change runtime type > T4 GPU`.

**No results are pre-filled in this notebook.** Every number, plot, and image you see is generated live when you run the cells — nothing here is fabricated or copied in from elsewhere.

## 1. Install / Import Dependencies

In [ ]:
!pip install -q scikit-image

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# All outputs from this notebook are written here, mirroring the project's outputs/ folder
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load MNIST and Create Noisy Versions

We add Gaussian noise (mean 0, controllable std-dev via `noise_factor`) and clip pixel values back to `[0, 1]`. This matches `src/data_utils.py` in the project exactly (same defaults, same seeded RNG for reproducibility).

In [ ]:
NOISE_FACTOR = 0.5
SEED = 42

(x_train, _), (x_test, y_test) = mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

rng = np.random.default_rng(SEED)
x_train_noisy = np.clip(x_train + NOISE_FACTOR * rng.normal(size=x_train.shape), 0., 1.).astype('float32')
x_test_noisy = np.clip(x_test + NOISE_FACTOR * rng.normal(size=x_test.shape), 0., 1.).astype('float32')

print('x_train:', x_train.shape, '| x_test:', x_test.shape)

In [ ]:
# Preview a few noisy vs clean pairs before training
n = 8
plt.figure(figsize=(16, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title('Clean')
    plt.axis('off')

    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_train_noisy[i].squeeze(), cmap='gray')
    plt.title('Noisy')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Train / Validation Split

In [ ]:
VAL_SPLIT = 0.1
n_val = int(len(x_train) * VAL_SPLIT)

x_val, x_val_noisy = x_train[:n_val], x_train_noisy[:n_val]
x_train_fit, x_train_noisy_fit = x_train[n_val:], x_train_noisy[n_val:]

print(f'Train: {x_train_fit.shape[0]} | Val: {x_val.shape[0]} | Test: {x_test.shape[0]}')

## 4. Build the Convolutional Autoencoder

Architecture matches `src/model.py` exactly.

In [ ]:
def build_autoencoder(input_shape=(28, 28, 1)):
    inp = layers.Input(shape=input_shape, name='noisy_input')

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    encoded = layers.MaxPooling2D((2, 2), padding='same', name='bottleneck')(x)

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoded)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoded = layers.Conv2D(input_shape[-1], (3, 3), activation='sigmoid', padding='same', name='denoised_output')(x)

    autoencoder = models.Model(inp, decoded, name='conv_autoencoder')
    autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
    return autoencoder

autoencoder = build_autoencoder()
autoencoder.summary()

## 5. Train the Model

Same hyperparameters and callbacks (early stopping, best-checkpoint saving) as `src/train.py`.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 128

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('denoising_autoencoder_best.h5', monitor='val_loss', save_best_only=True, verbose=1),
]

history = autoencoder.fit(
    x_train_noisy_fit, x_train_fit,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_val_noisy, x_val),
    callbacks=callbacks,
    verbose=2
)

autoencoder.save('denoising_autoencoder_final.h5')
print('Model saved: denoising_autoencoder_final.h5')

## 6. Plot and Save Training / Validation Loss

Saved to `outputs/loss_curve.png`, matching what `src/train.py` produces locally.

In [ ]:
plt.figure()
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Crossentropy)')
plt.title('Training vs Validation Loss')
plt.legend()
loss_curve_path = os.path.join(OUTPUT_DIR, 'loss_curve.png')
plt.savefig(loss_curve_path)
plt.show()
print('Saved:', loss_curve_path)

## 7. Evaluate: MSE, PSNR, SSIM

We compare the **baseline** (noisy vs. original, i.e. no denoising) against the **model** (denoised vs. original) on the full 10,000-image MNIST test set. This mirrors `src/metrics.py` and `src/evaluate.py` exactly, so results are consistent whether you train here or locally.

In [ ]:
def compute_metrics(originals, reconstructions):
    mse_list, psnr_list, ssim_list = [], [], []
    for orig, recon in zip(originals, reconstructions):
        o, r = orig.squeeze(), recon.squeeze()
        mse_list.append(float(np.mean((o - r) ** 2)))
        psnr_list.append(float(psnr(o, r, data_range=1.0)))
        ssim_list.append(float(ssim(o, r, data_range=1.0)))
    return {
        'mse': float(np.mean(mse_list)),
        'psnr': float(np.mean(psnr_list)),
        'ssim': float(np.mean(ssim_list)),
    }

denoised = autoencoder.predict(x_test_noisy)

metrics_baseline = compute_metrics(x_test, x_test_noisy)
metrics_model = compute_metrics(x_test, denoised)

print('Baseline (noisy vs original, no denoising):', metrics_baseline)
print('Model    (denoised vs original):', metrics_model)

metrics_path = os.path.join(OUTPUT_DIR, 'metrics.txt')
with open(metrics_path, 'w') as f:
    f.write('Evaluated on MNIST test set (10,000 images)\n\n')
    f.write('Baseline (noisy vs original, i.e. no denoising applied):\n')
    f.write(str(metrics_baseline) + '\n\n')
    f.write('Model (denoised vs original):\n')
    f.write(str(metrics_model) + '\n')
print('Saved:', metrics_path)

## 8. Visualize and Save: Original vs Noisy vs Denoised

Saved to `outputs/sample_results.png`.

In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap='gray')
    plt.title('Original')
    plt.axis('off')

    plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap='gray')
    plt.title('Noisy')
    plt.axis('off')

    plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised[i].squeeze(), cmap='gray')
    plt.title('Denoised')
    plt.axis('off')
plt.tight_layout()
sample_results_path = os.path.join(OUTPUT_DIR, 'sample_results.png')
plt.savefig(sample_results_path)
plt.show()
print('Saved:', sample_results_path)

## 9. Download Everything

This packages the trained model and all evaluation outputs (`loss_curve.png`, `sample_results.png`, `metrics.txt`) into one zip so you can bring real, non-fabricated results back into the local project folders:
- put `denoising_autoencoder_final.h5` into `saved_model/`
- put the three files from `outputs/` into the project's `outputs/` folder

This is also what you'll want for your project report (Section 8 of `report/Project_Report_Outline.md`).

In [ ]:
import shutil
from google.colab import files

bundle_dir = 'colab_run_results'
os.makedirs(bundle_dir, exist_ok=True)
shutil.copy('denoising_autoencoder_final.h5', bundle_dir)
shutil.copy(os.path.join(OUTPUT_DIR, 'loss_curve.png'), bundle_dir)
shutil.copy(os.path.join(OUTPUT_DIR, 'sample_results.png'), bundle_dir)
shutil.copy(os.path.join(OUTPUT_DIR, 'metrics.txt'), bundle_dir)

zip_path = shutil.make_archive('colab_run_results', 'zip', bundle_dir)
print('Created:', zip_path)

files.download(zip_path)

## Notes
- This notebook is self-contained: it does not import from `src/`, so it works in a fresh Colab session with nothing else uploaded — every cell above must be run in order, top to bottom, for the later cells to work.
- The model architecture, noise generation, hyperparameters, and metrics computation all mirror `src/model.py`, `src/data_utils.py`, `src/train.py`, `src/evaluate.py`, and `src/metrics.py` from the local project, so results are consistent whether you train here or locally.
- Everything above runs live in Colab — there are no pre-filled or fabricated results in this notebook.
- To experiment: try different `NOISE_FACTOR` values, more epochs, or extra Conv layers, and see how PSNR/SSIM change.